In [1]:
import os
import dotenv 

dotenv.load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_core.documents import Document

documents = [
  Document(
    page_content="Dogs are great companions, knowm for their loyalty and friendliness",
    metadata={"source": "mammal-pets-doc"}
  ),
  Document(
    page_content="Cats are independent pets and often enjoy their own space",
    metadata={"source": "mammal-pets-doc"}
  ),
  Document(
    page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
    metadata={"source": "mammal-pets-doc"}
  ),
]

In [3]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, knowm for their loyalty and friendliness'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets and often enjoy their own space'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')]

In [4]:
# Vector Store
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

model = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

c:\Users\anand\Desktop\GenAI - Krish Naik\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-mpnet-base-v2")

In [6]:
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings)

In [8]:
vector_store.similarity_search("goldfish")

[Document(id='b20e33e4-c45d-4676-9f90-bc34b4230073', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(id='031c6030-ccfa-4f98-abce-22f97d13f48a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, knowm for their loyalty and friendliness'),
 Document(id='2d23d7c6-6597-42a6-9706-ef9b12cf6625', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets and often enjoy their own space')]

In [9]:
vector_store.similarity_search_with_score("dog")

[(Document(id='031c6030-ccfa-4f98-abce-22f97d13f48a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, knowm for their loyalty and friendliness'),
  0.936769962310791),
 (Document(id='b20e33e4-c45d-4676-9f90-bc34b4230073', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
  1.6417516469955444),
 (Document(id='2d23d7c6-6597-42a6-9706-ef9b12cf6625', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets and often enjoy their own space'),
  1.6523357629776)]

In [12]:
from typing import List
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(["goldfish", "cat"])

[[Document(id='b20e33e4-c45d-4676-9f90-bc34b4230073', metadata={'source': 'mammal-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.')],
 [Document(id='2d23d7c6-6597-42a6-9706-ef9b12cf6625', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets and often enjoy their own space')]]

In [13]:
# 2nd method - using as_retriever method

retriever = vector_store.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 1}
)

retriever.batch(["cat", "dog"])

[[Document(id='2d23d7c6-6597-42a6-9706-ef9b12cf6625', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets and often enjoy their own space')],
 [Document(id='031c6030-ccfa-4f98-abce-22f97d13f48a', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, knowm for their loyalty and friendliness')]]